# Remaining position-template coverage

This notebook audits template availability for goalkeepers, midfield roles, wingers and forwards. It records where the five-appearance rule and the requirement for at least two qualifying players permit a defensible league template.

Run `scripts/build_remaining_position_templates.py` first. Generated outputs remain local because they derive from licensed event data.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'

coverage = pd.read_csv(PROCESSED_ROOT / 'remaining_position_template_coverage.csv')
coverage = coverage[[
    'position_group', 'template_group', 'template_scope',
    'player_profiles', 'qualifying_players', 'template_available'
]]

## 1. League template availability

A template is available only when at least two players have five eligible appearances in that exact group. This ensures a qualifying player can be compared with an independent leave-one-out baseline.

In [ ]:
coverage.sort_values(['position_group', 'template_scope', 'template_group'])

In [ ]:
coverage.loc[~coverage['template_available']]

## 2. Nottingham Forest player coverage

The table below shows Forest profiles that can eventually be compared with each available league group. Player eligibility is evaluated separately for every role-specific and overarching group.

In [ ]:
stems = [
    'goalkeeper', 'defensive_midfield', 'central_midfield',
    'attacking_midfield', 'wide_midfield', 'winger', 'forward'
]
forest_rows = []
for stem in stems:
    profiles = pd.read_parquet(PROCESSED_ROOT / f'{stem}_player_zone_profiles.parquet')
    forest = profiles.loc[
        profiles['teams'].str.contains('Nottingham Forest', na=False),
        ['template_group', 'player_name', 'eligible_appearances', 'template_eligible'],
    ].drop_duplicates()
    forest['position_group'] = stem.replace('_', ' ').title()
    forest_rows.append(forest)
forest_coverage = pd.concat(forest_rows, ignore_index=True)
forest_coverage.sort_values(
    ['position_group', 'template_group', 'eligible_appearances'],
    ascending=[True, True, False],
)

In [ ]:
forest_coverage.loc[forest_coverage['template_eligible']].sort_values(
    ['position_group', 'template_group', 'eligible_appearances'],
    ascending=[True, True, False],
)

## Decision

Goalkeeper, central-midfield, attacking-midfield, wide-midfield, winger and forward league templates have sufficient peer coverage to proceed to formal validation. Defensive midfield does not: only one player reaches five overarching appearances, while the left and right defensive-midfield roles have none. The defensive-midfield profiles should remain exploratory until more data or a revised role-mapping strategy provides a viable comparison population.

For Forest, the strongest immediate samples are goalkeeper, central midfield, attacking midfield and centre-forward. Forest wide-midfield and winger appearances are currently provisional despite the league templates being available.